In [1]:
import os
import gzip
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import anndata as ad

from rds2py import read_rds


In [ ]:
from pathlib import Path
import pandas as pd

rds_path = Path("../data/211028_retina_cells_annotated_220708.RDS")

def load_rds(path: Path):
    # 1) Fast path: works when the RDS stores a plain data.frame/tibble
    try:
        import pyreadr
        result = pyreadr.read_r(str(path))
        df = next(iter(result.values()))
        return df, "pyreadr"
    except Exception as e_pyreadr:
        pyreadr_error = e_pyreadr

    # 2) Fallback: use rpy2 for complex R objects (e.g., Seurat/SCE)
    try:
        import rpy2.robjects as ro
        from rpy2.robjects import pandas2ri
        from rpy2.robjects.conversion import localconverter

        ro.globalenv["rds_path"] = str(path)
        ro.r("obj <- readRDS(rds_path)")
        r_classes = [str(x) for x in ro.r("class(obj)")]

        if "Seurat" in r_classes:
            ro.r("df <- obj@meta.data")
            source = "rpy2 (Seurat meta.data)"
        else:
            ro.r("df <- tryCatch(as.data.frame(obj), error=function(e) NULL)")
            is_null = bool(ro.r("is.null(df)")[0])
            if is_null:
                raise TypeError(f"R object class {r_classes} cannot be converted to pandas DataFrame directly.")
            source = f"rpy2 (as.data.frame), class={r_classes}"

        with localconverter(ro.default_converter + pandas2ri.converter):
            df = ro.conversion.rpy2py(ro.r("df"))

        return df, source

    except Exception as e_rpy2:
        raise RuntimeError(
            "Could not load the RDS file with either pyreadr or rpy2.\n"
            f"pyreadr error: {type(pyreadr_error).__name__}: {pyreadr_error}\n"
            f"rpy2 error: {type(e_rpy2).__name__}: {e_rpy2}\n\n"
            "If needed, install fallback deps: pip install rpy2"
        )

retina_df, loader = load_rds(rds_path)

print(f"Loaded: {rds_path}")
print(f"Loader: {loader}")
print(f"Shape: {retina_df.shape}")
retina_df.head()

LibrdataError: The file contains an unrecognized object